# Claims Aging Pipeline
**Closed Summary · Open Aging Pareto · Open Tasks**

Single-entry pipeline: `data/raw → interim → processed → exports`  
All business logic lives in `src/`. This notebook is the orchestrator.

In [ ]:
# ── STEP 1 / 8 — Imports ────────────────────────────────────────────────
import sys
from pathlib import Path

# Add project root to sys.path so `src` is importable from notebooks/
_root = Path.cwd()
for _p in [_root, *_root.parents]:
    if (_p / "src").is_dir():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break

import yaml
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')          # non-interactive backend for saved PNGs
import matplotlib.pyplot as plt

from src.paths      import get_paths
from src.loaders    import load_excel, normalize_columns
from src.transforms import (
    filter_by_warehouses_and_dates,
    closed_claims_summary,
    open_claims_aging_table,
    build_aging_summary,
    build_open_tasks,
)
from src.exporters  import plot_pareto_for_wh, export_claims_workbook

print("[STEP 1/8] Imports complete")

In [ ]:
# ── STEP 2 / 8 — Load Configuration ─────────────────────────────────────
P = get_paths()

with open(P['config'] / 'settings.yaml') as f:
    cfg = yaml.safe_load(f)

warehouses   = cfg['warehouses']
start_date   = cfg['start_date']
end_date     = cfg['end_date']
max_labels   = cfg['max_labels']
input_files  = [(item['path'], item['name']) for item in cfg['input_files']]
output_excel = cfg['output_excel']

print(f"[STEP 2/8] Config loaded — {len(warehouses)} warehouses, "
      f"date range {start_date} → {end_date}, "
      f"{len(input_files)} input file(s)")

In [ ]:
# ── STEP 3 / 8 — Resolve Paths ──────────────────────────────────────────
# P was initialized in Step 2 alongside config.
# Confirm all required directories exist.
for key in ('raw', 'exports', 'charts'):
    P[key].mkdir(parents=True, exist_ok=True)

print(f"[STEP 3/8] Paths resolved — root: {P['root']}")
print(f"           raw:     {P['raw']}")
print(f"           exports: {P['exports']}")
print(f"           charts:  {P['charts']}")

In [ ]:
# ── STEP 4 / 8 — Load Data ──────────────────────────────────────────────
raw_frames = {}   # dataset_name → raw DataFrame

for rel_path, dname in input_files:
    full_path = P['raw'] / rel_path
    if not full_path.exists():
        print(f"  WARNING: File not found, skipping: {full_path}")
        continue
    raw_frames[dname] = load_excel(full_path)
    print(f"  Loaded '{dname}' — {len(raw_frames[dname]):,} rows from {full_path.name}")

print(f"[STEP 4/8] Load complete — {len(raw_frames)} dataset(s) loaded")

In [ ]:
# ── STEP 5 / 8 — Clean / Validate ───────────────────────────────────────
# normalize_columns() standardizes column names and casts types.
# filter_by_warehouses_and_dates() applies warehouse list + date range.

normalized  = {}   # dataset_name → (df_std, df_filtered)
combined_rows = []

for dname, raw_df in raw_frames.items():
    df_std, rename_map = normalize_columns(raw_df)

    # Merge standardized columns back onto original (preserves extra columns)
    df_comb = raw_df.copy()
    for col in ['Warehouse', 'Start Date', 'Completed Date',
                'Task Name', 'Labels', 'Due Date']:
        if col in df_std.columns:
            df_comb[col] = df_std[col]
    df_comb['Dataset'] = dname
    combined_rows.append(df_comb)

    df_filt = filter_by_warehouses_and_dates(df_std, warehouses, start_date, end_date)
    normalized[dname] = (df_std, df_filt)
    print(f"  '{dname}': {len(df_std):,} rows normalized → {len(df_filt):,} in scope")

print(f"[STEP 5/8] Clean/Validate complete — {len(normalized)} dataset(s) processed")

In [ ]:
%%time
# ── STEP 6 / 8 — Transform / Analyze ────────────────────────────────────

closed_summaries   = []
open_aging_tables  = []
open_task_rows     = []

for dname, (df_std, df_filt) in normalized.items():
    closed_summaries.append(closed_claims_summary(df_filt, dname))
    open_aging_tables.append(open_claims_aging_table(df_filt, dname, warehouses))
    open_task_rows.append(build_open_tasks(df_filt, dname))

# Combined source rows (all datasets, filtered)
combined_all = pd.concat(combined_rows, ignore_index=True) if combined_rows else pd.DataFrame()
if not combined_all.empty:
    mask = combined_all['Warehouse'].notna()
    if start_date:
        mask &= pd.to_datetime(combined_all['Start Date'], errors='coerce') >= pd.to_datetime(start_date)
    if end_date:
        mask &= pd.to_datetime(combined_all['Start Date'], errors='coerce') <= pd.to_datetime(end_date)
    wanted = {w.strip().lower() for w in warehouses}
    mask &= combined_all['Warehouse'].astype(str).str.lower().isin(wanted)
    combined_filtered = combined_all[mask].copy()
else:
    combined_filtered = combined_all

closed_all = pd.concat(closed_summaries,  ignore_index=True) if closed_summaries  else pd.DataFrame()
aging_all  = pd.concat(open_aging_tables, ignore_index=True) if open_aging_tables else pd.DataFrame()

aging_summary = build_aging_summary(aging_all) if not aging_all.empty else pd.DataFrame()

open_tasks_detail = (
    pd.concat(open_task_rows, ignore_index=True)
    .sort_values(['Action Required', 'Due Date', 'Warehouse'], na_position='last')
    .reset_index(drop=True)
) if open_task_rows else pd.DataFrame(
    columns=['Dataset', 'Warehouse', 'Task Name', 'Action Required', 'Due Date']
)

open_tasks_agg = (
    open_tasks_detail
    .groupby(['Dataset', 'Warehouse', 'Action Required'], dropna=False)
    .size().rename('Count').reset_index()
    .sort_values(['Warehouse', 'Count'], ascending=[True, False], ignore_index=True)
) if not open_tasks_detail.empty else pd.DataFrame(
    columns=['Dataset', 'Warehouse', 'Action Required', 'Count']
)

print(f"[STEP 6/8] Transform complete")
print(f"           combined_filtered: {len(combined_filtered):,} rows")
print(f"           closed_all:        {len(closed_all):,} rows")
print(f"           aging_all:         {len(aging_all):,} rows")
print(f"           open_tasks_detail: {len(open_tasks_detail):,} rows")

In [ ]:
# ── STEP 7 / 8 — Visualize ──────────────────────────────────────────────
# Pareto charts saved to charts/ (one PNG per dataset × warehouse).

chart_count = 0

if not aging_all.empty:
    desired_order = ['<30 days', '30-<60 days', '60-<90 days', '>= 90 days']
    for dname in aging_all['Dataset'].unique():
        sub = aging_all[aging_all['Dataset'] == dname]
        for wh in warehouses:
            wh_rows = sub[sub['Warehouse'] == wh]
            if wh_rows.empty:
                continue
            wh_rows = (
                wh_rows.set_index('Aging Bucket')
                .reindex(desired_order)
                .reset_index()
            )
            plot_pareto_for_wh(
                wh_rows, dname, wh,
                max_labels=max_labels,
                charts_path=P['charts'],
            )
            chart_count += 1
else:
    print('  No aging data — charts skipped.')

print(f"[STEP 7/8] Visualize complete — {chart_count} chart(s) saved to {P['charts']}")

In [ ]:
# ── STEP 8 / 8 — Export Outputs ─────────────────────────────────────────
# Single multi-sheet workbook written to data/exports/.

out_path = export_claims_workbook(
    combined_filtered=combined_filtered,
    closed_all=closed_all,
    aging_all=aging_all,
    aging_summary=aging_summary,
    open_tasks_detail=open_tasks_detail,
    open_tasks_agg=open_tasks_agg,
    exports_path=P['exports'],
    filename=output_excel,
)

print(f"[STEP 8/8] Export complete")
print(f"           Workbook: {out_path}")
print(f"           Sheets  : All_Claims_Combined, Closed_Claims_Summary, "
      f"Open_Claims_Aging, Open_Aging_Summary, "
      f"Open_Tasks_By_Warehouse, Open_Tasks_Aggregate")

In [ ]:
# ── Preview — Aging Summary (leadership view) ────────────────────────────
if not aging_summary.empty:
    display(
        aging_summary
        .sort_values(['Dataset', 'Total Open Claims'], ascending=[True, False])
        .reset_index(drop=True)
        .style.format('{:.1%}', subset=['% >= 90 days'])
    )